# Phase1_Data_Exploration
**Course:** SWE 485  

## Dataset Goal & Source
**Goal:** Identify patterns that indicate higher depression risk among students. <br>
**Source:** https://www.kaggle.com/datasets/aldinwhyudii/student-depression-and-lifestyle-100k-data?resource=download<br>
**Justification:** This dataset was selected because it includes multiple variables describing students’ academic conditions and daily lifestyle habits. These factors provide meaningful information that allows us to analyze how different aspects of students’ lives may influence depression risk and support the development of models that estimate this risk.


# Exploratory Data Analysis — Student Lifestyle & Depression Dataset

**Dataset:** `student_lifestyle_100k.csv`  
**Observations:** 100,000  
**Target Variable:** `Depression` (Boolean — Binary Classification)  

In [ ]:
import sys
print(sys.executable)
!{sys.executable} -m pip install pandas numpy matplotlib seaborn scikit-learn jupyter

## 1. Setup & Imports <a id='1'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Global plot settings
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.dpi': 120,
})

COLORS = {'no_dep': '#4CAF50', 'dep': '#F44336', 'blue': '#1F4E79', 'accent': '#2E86AB'}


## 2. Load & Inspect Data <a id='2'></a>

In [ ]:
df = pd.read_csv('Dataset/student_lifestyle_100k.csv')

print(f'Shape: {df.shape}')
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
df.head(10)

In [ ]:
df.dtypes

## 3. Dataset Overview <a id='3'></a>

In [ ]:
# Statistical Summary
summary = df.describe().round(3)
print(summary)



In [ ]:
# Categorical features summary
print('--- Gender Distribution ---')
print(df['Gender'].value_counts())
print()
print('--- Department Distribution ---')
print(df['Department'].value_counts())
print()
print('--- Depression Distribution ---')
print(df['Depression'].value_counts())
print(f'\nDepression Rate: {df["Depression"].mean()*100:.2f}%')

## 4. Missing Value Analysis <a id='4'></a>

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})

print(missing_df)

print('\nNo Missing Values!\nDataset is 100% Complete')



## 5. Target Variable — Class Distribution <a id='5'></a>

In [ ]:
dep_counts = df['Depression'].value_counts()
labels = ['Not Depressed (False)', 'Depressed (True)']
colors = [COLORS['no_dep'], COLORS['dep']]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Target Variable: Depression Distribution', fontsize=16, fontweight='bold', color=COLORS['blue'])

# Pie chart
wedges, texts, autotexts = axes[0].pie(
    dep_counts, labels=labels, autopct='%1.2f%%',
    colors=colors, startangle=90, pctdistance=0.75,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')
axes[0].set_title('Proportion', fontweight='bold')

# Bar chart
bars = axes[1].bar(labels, dep_counts.values, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
axes[1].set_title('Absolute Count', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].set_xticklabels(labels, rotation=10, ha='right')
for bar, val in zip(bars, dep_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 400,
                 f'{val:,}', ha='center', fontsize=11, fontweight='bold')

# Horizontal bar
axes[2].barh(labels, dep_counts.values, color=colors, edgecolor='white', height=0.4)
axes[2].set_title('Horizontal View', fontweight='bold')
axes[2].set_xlabel('Count')
for i, val in enumerate(dep_counts.values):
    axes[2].text(val + 500, i, f'{val:,}', va='center', fontsize=11)

plt.tight_layout()
plt.show()

print(f'\nClass Imbalance Ratio: {dep_counts[False]/dep_counts[True]:.1f}:1 (Not Depressed : Depressed)')

## 6. Feature Distributions — Histograms <a id='6'></a>

In [ ]:
num_cols = ['Age', 'CGPA', 'Sleep_Duration', 'Study_Hours',
            'Social_Media_Hours', 'Physical_Activity', 'Stress_Level']

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('Feature Distributions', fontsize=16, fontweight='bold', color=COLORS['blue'], y=1.01)
axes = axes.flatten()

for i, col in enumerate(num_cols):
    # Overlay depressed vs non-depressed
    axes[i].hist(df[df['Depression']==False][col], bins=30, alpha=0.6,
                 color=COLORS['no_dep'], label='Not Depressed', edgecolor='white')
    axes[i].hist(df[df['Depression']==True][col], bins=30, alpha=0.6,
                 color=COLORS['dep'], label='Depressed', edgecolor='white')
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].legend(fontsize=8)

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

## 7. Box Plots by Depression Status <a id='7'></a>

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('Feature Distributions by Depression Status', fontsize=16,
             fontweight='bold', color=COLORS['blue'], y=1.01)
axes = axes.flatten()

for i, col in enumerate(num_cols):
    data_false = df[df['Depression']==False][col]
    data_true  = df[df['Depression']==True][col]
    bp = axes[i].boxplot(
        [data_false, data_true],
        patch_artist=True,
        labels=['Not Depressed', 'Depressed'],
        medianprops={'color':'black','linewidth':2}
    )
    bp['boxes'][0].set_facecolor(COLORS['no_dep'])
    bp['boxes'][0].set_alpha(0.7)
    bp['boxes'][1].set_facecolor(COLORS['dep'])
    bp['boxes'][1].set_alpha(0.7)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel(col)

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

## 8. Categorical Feature Analysis <a id='8'></a>

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Categorical Feature Analysis', fontsize=16, fontweight='bold', color=COLORS['blue'])

# Gender distribution
gender_counts = df['Gender'].value_counts()
axes[0,0].bar(gender_counts.index, gender_counts.values,
              color=[COLORS['accent'],'#E91E63'], edgecolor='white', width=0.5)
axes[0,0].set_title('Gender Distribution', fontweight='bold')
axes[0,0].set_ylabel('Count')
for i, v in enumerate(gender_counts.values):
    axes[0,0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

# Department distribution
dept_counts = df['Department'].value_counts()
axes[0,1].barh(dept_counts.index, dept_counts.values,
               color=sns.color_palette('muted', len(dept_counts)), edgecolor='white')
axes[0,1].set_title('Department Distribution', fontweight='bold')
axes[0,1].set_xlabel('Count')
for i, v in enumerate(dept_counts.values):
    axes[0,1].text(v + 100, i, f'{v:,}', va='center', fontsize=10)

# Depression by Gender
gender_dep = df.groupby(['Gender','Depression']).size().unstack(fill_value=0)
x = np.arange(len(gender_dep.index))
w = 0.35
axes[1,0].bar(x - w/2, gender_dep[False], w, label='Not Depressed', color=COLORS['no_dep'], edgecolor='white')
axes[1,0].bar(x + w/2, gender_dep[True],  w, label='Depressed',     color=COLORS['dep'],    edgecolor='white')
axes[1,0].set_xticks(x)
axes[1,0].set_xticklabels(gender_dep.index)
axes[1,0].set_title('Depression by Gender', fontweight='bold')
axes[1,0].set_ylabel('Count')
axes[1,0].legend()

# Depression rate by Department
dept_dep_rate = df.groupby('Department')['Depression'].mean() * 100
bars = axes[1,1].bar(dept_dep_rate.index, dept_dep_rate.values,
                     color=sns.color_palette('Reds_d', len(dept_dep_rate)), edgecolor='white')
axes[1,1].set_title('Depression Rate (%) by Department', fontweight='bold')
axes[1,1].set_ylabel('Depression Rate (%)')
axes[1,1].set_xticklabels(dept_dep_rate.index, rotation=15, ha='right')
for bar, val in zip(bars, dept_dep_rate.values):
    axes[1,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                   f'{val:.1f}%', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 9. Correlation Heatmap <a id='9'></a>

In [ ]:
df_enc = df.copy()
df_enc['Depression_num'] = df_enc['Depression'].astype(int)
df_enc['Gender_num']     = (df_enc['Gender'] == 'Male').astype(int)

corr_cols = ['Age','CGPA','Sleep_Duration','Study_Hours','Social_Media_Hours',
             'Physical_Activity','Stress_Level','Gender_num','Depression_num']
corr = df_enc[corr_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Correlation Analysis', fontsize=16, fontweight='bold', color=COLORS['blue'])

# Full heatmap
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=axes[0], linewidths=0.5, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 9})
axes[0].set_title('Full Correlation Heatmap', fontweight='bold')

# Correlation with Depression only
dep_corr = corr['Depression_num'].drop('Depression_num').sort_values()
colors_bar = [COLORS['dep'] if v > 0 else COLORS['no_dep'] for v in dep_corr.values]
bars = axes[1].barh(dep_corr.index, dep_corr.values, color=colors_bar, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('Correlation with Depression', fontweight='bold')
axes[1].set_xlabel('Correlation Coefficient')
for bar, val in zip(bars, dep_corr.values):
    xpos = val + 0.003 if val >= 0 else val - 0.003
    ha = 'left' if val >= 0 else 'right'
    axes[1].text(xpos, bar.get_y()+bar.get_height()/2,
                 f'{val:.3f}', va='center', ha=ha, fontsize=10)

patch_pos = mpatches.Patch(color=COLORS['dep'], label='Positive correlation')
patch_neg = mpatches.Patch(color=COLORS['no_dep'], label='Negative correlation')
axes[1].legend(handles=[patch_pos, patch_neg], loc='lower right')

plt.tight_layout()
plt.show()

## 10. Feature Importance (Preliminary) <a id='12'></a>

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

df_ml = df.copy()
df_ml['Gender']     = LabelEncoder().fit_transform(df_ml['Gender'])
df_ml['Department'] = LabelEncoder().fit_transform(df_ml['Department'])
df_ml['Depression'] = df_ml['Depression'].astype(int)

features = ['Age','Gender','Department','CGPA','Sleep_Duration','Study_Hours',
            'Social_Media_Hours','Physical_Activity','Stress_Level']

X = df_ml[features]
y = df_ml['Depression']

# Train a quick random forest for importance
rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors_fi = sns.color_palette('RdYlGn', len(importances))
bars = ax.barh(importances.index, importances.values, color=colors_fi, edgecolor='white')
ax.set_title('Feature Importance (Random Forest — Preliminary)',
             fontsize=14, fontweight='bold', color=COLORS['blue'])
ax.set_xlabel('Importance Score')
for bar, val in zip(bars, importances.values):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print('\nTop 3 Most Important Features:')
for feat, score in importances.sort_values(ascending=False).head(3).items():
    print(f'  {feat}: {score:.4f}')

## 13. Key Insights Summary <a id='13'></a>

In [ ]:
print('=' * 65)
print('         EDA SUMMARY — STUDENT LIFESTYLE DATASET')
print('=' * 65)
print(f'Total Records       : {len(df):,}')
print(f'Total Features      : {df.shape[1]}')
print(f'Missing Values      : {df.isnull().sum().sum()}')
print(f'Depressed Students  : {df["Depression"].sum():,} ({df["Depression"].mean()*100:.2f}%)')
print(f'Class Imbalance     : {(~df["Depression"]).sum()/(df["Depression"].sum()):.1f}:1')
print()
print('--- Top Correlations with Depression ---')
dep_corr_sorted = df_enc[corr_cols].corr()['Depression_num'].drop('Depression_num').sort_values(key=abs, ascending=False)
for feat, val in dep_corr_sorted.items():
    direction = '↑ Positive' if val > 0 else '↓ Negative'
    print(f'  {feat:<25} r = {val:+.3f}  {direction}')
print()
print('--- Key Takeaways ---')
print('  1. CGPA is the strongest predictor of depression ')
print('  2. Stress_Level rises sharply to near 100% depression at level 10')
print('  3. Sleep_Duration protects against depression ')
print('  4. Dataset has significant class imbalance ')
print('  5. No missing values ')
print('=' * 65)

# Data Preprocessing & Feature Engineering

## Cleaning the dataset

In [ ]:
#First, we copy the original dataset so we can start processing
df_processed = df.copy()

#Now, we remove irrelevant columns
df_processed = df_processed.drop("Student_ID", axis=1)

## Encoding categorical variables

In [ ]:
#Since these categorical values have no inherent order, one hot encoding is suitable as it treats each category independently.
#We encode to convert strings to a format the model can understand.
categories = ["Gender", "Department"]
df_processed = pd.get_dummies(df_processed, columns=categories)
df = df_processed.copy()

## Advanced Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# -----------------------------
# Handle outliers using IQR method
# -----------------------------
numeric_cols = df.select_dtypes(include=np.number).columns.drop('Student_ID')  # numeric columns only, keep IDs intact

Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

# keep only rows within 1.5*IQR
df = df[~((df[numeric_cols] < (Q1 - 1.5 * IQR)) | (df[numeric_cols] > (Q3 + 1.5 * IQR))).any(axis=1)]

# -----------------------------
# Remove highly correlated features
# -----------------------------
corr_matrix = df[numeric_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
df = df.drop(columns=to_drop)

# -----------------------------
# Handle skewed distributions (log transform for positive numeric features)
# -----------------------------
positive_cols = [col for col in numeric_cols if (df[col] > 0).all()]
for col in positive_cols:
    df[col] = np.log1p(df[col])

# -----------------------------
# Create new engineered features
# -----------------------------
df['Total_Activity'] = df['Study_Hours'] + df['Social_Media_Hours']
df['Sleep_Study_Ratio'] = df['Sleep_Duration'] / (df['Study_Hours'] + 1)
df['Activity_Per_Hour'] = df['Total_Activity'] / (df['Study_Hours'] + df['Social_Media_Hours'] + 1)
df['Sleep_Deficit'] = 8 - df['Sleep_Duration']  # ideal sleep assumption

# -----------------------------
# Normalize / scale numeric features
# -----------------------------
numeric_cols = df.select_dtypes(include=np.number).columns.drop('Student_ID')  # exclude ID from scaling
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# -----------------------------
# Show the first few rows
# -----------------------------
df.head()

In [ ]:
df.to_csv('clean_data.csv', index=False)